# Populate the Amazon S3 Vectors index with Nova 2 multimodal embeddings

This notebook creates and populates the vector index that the in-process Strands fashion agent queries for image lookup. It embeds a demo fashion dataset with the **Amazon Nova 2 multimodal embeddings** model and writes the vectors (plus a catalog `s3_uri` in the metadata) to an **Amazon S3 Vectors** index.

**Important (two-step provisioning):** the vector *bucket* is created by `cdk deploy` (the CDK data-plane stack), but the vector *index* is created **here**. The application refuses to start until the index exists, so you must run this notebook **once** after deploying and before launching the Streamlit app.

No search-cluster dependency, no AWS profile name, and no base64 blob is stored in the vector metadata (only the catalog `s3_uri` + `name`). Credentials come from the default boto3 chain.

## 1. Download the demo dataset

For demo purposes we use an external fashion image dataset from GitHub.

In [ ]:
!git clone https://github.com/orbitalsonic/Fashion-Dataset-Images-Western-Dress.git

## 2. Install dependencies and import

Install the project (which pins an s3vectors-capable boto3) in editable mode, then import.

In [ ]:
!pip install -e '.[infra,dev]'

In [ ]:
import base64
import io
import json
import os
from pathlib import Path

import boto3
import yaml
from botocore.exceptions import ClientError
from PIL import Image
from tqdm.auto import tqdm

## 3. Load config and CDK outputs

`config.yml` supplies the embedding model id, the embedding region, and the expected dimension; `variables.json` (written by `cdk deploy --outputs-file variables.json`) supplies the vector bucket + index names and the image bucket.

In [ ]:
ROOT = Path(os.path.abspath(""))

with open(ROOT / "config.yml", "r") as fh:
    config = yaml.safe_load(fh)

with open(ROOT / "variables.json", "r") as fh:
    raw_outputs = json.load(fh)


def flatten_outputs(raw):
    """Accept either flat CDK outputs or the {StackName: {...}} nested shape."""
    if any(k in raw for k in ("BucketName", "VectorBucketName", "VectorIndexName")):
        return raw
    flat = {}
    for value in raw.values():
        if isinstance(value, dict):
            flat.update(value)
    return flat


outputs = flatten_outputs(raw_outputs)

IMAGE_BUCKET = outputs["BucketName"]
VECTOR_BUCKET = outputs["VectorBucketName"]
VECTOR_INDEX = outputs["VectorIndexName"]

EMBEDDING_REGION = config["embedding_region"]
PRIMARY_REGION = config["primary_region"]
EMBEDDING_MODEL_ID = config["models"]["embeddings"]
EXPECTED_DIMENSION = config["vector_index"].get("expected_dimension")  # may be None
DISTANCE_METRIC = config["vector_index"].get("distance_metric", "cosine")

print(f"image bucket:  {IMAGE_BUCKET}")
print(f"vector bucket: {VECTOR_BUCKET}")
print(f"vector index:  {VECTOR_INDEX}")
print(f"embedding model: {EMBEDDING_MODEL_ID} @ {EMBEDDING_REGION}")
print(f"expected_dimension (config): {EXPECTED_DIMENSION}")

In [ ]:
# Default credential chain (NFR-1): no profile_name, no hardcoded creds.
session = boto3.Session()
bedrock = session.client("bedrock-runtime", region_name=EMBEDDING_REGION)
s3 = session.client("s3", region_name=PRIMARY_REGION)
s3vectors = session.client("s3vectors", region_name=PRIMARY_REGION)

## 4. Nova 2 embedding helper

This mirrors `components/agent/tools/embeddings.py`: the `[DOC]` Nova 2 `SINGLE_EMBEDDING` body with an `embeddingPurpose`. The `embeddingDimension` hint is sent **only** when `config.vector_index.expected_dimension` is non-null (LOCKED FIX 3 — never hardcode 1024). The vector is read from `embeddings[0].embedding`.

In [ ]:
PURPOSE_INDEX = "GENERIC_INDEX"  # [DOC] embeddingPurpose for ingestion
TASK_TYPE = "SINGLE_EMBEDDING"   # [DOC]
IMAGE_FORMAT = "jpeg"            # [DOC]


def embed(text=None, image_b64=None, purpose=PURPOSE_INDEX):
    if not text and not image_b64:
        raise ValueError("embed requires text and/or image_b64")
    params = {"embeddingPurpose": purpose}
    # LOCKED FIX 3: send the dimension hint only when expected_dimension is set.
    if EXPECTED_DIMENSION is not None:
        params["embeddingDimension"] = int(EXPECTED_DIMENSION)
    if text:
        params["text"] = {"truncationMode": "END", "value": text}
    if image_b64:
        params["image"] = {"format": IMAGE_FORMAT, "source": {"bytes": image_b64}}
    body = json.dumps({"taskType": TASK_TYPE, "singleEmbeddingParams": params})
    response = bedrock.invoke_model(
        modelId=EMBEDDING_MODEL_ID,
        body=body,
        accept="application/json",
        contentType="application/json",
    )
    payload = json.loads(response["body"].read())
    return payload["embeddings"][0]["embedding"]


def encode_image(image_path, max_size=(1024, 1024)):
    """Resize-if-large and return base64 JPEG bytes for embedding."""
    with Image.open(image_path) as image:
        image = image.convert("RGB")
        if image.size[0] * image.size[1] > max_size[0] * max_size[1]:
            image.thumbnail(max_size)
        buf = io.BytesIO()
        image.save(buf, format="JPEG")
    return base64.b64encode(buf.getvalue()).decode("utf8")

## 5. Bootstrap the index (the ONLY `create_index` call site)

LOCKED FIX 2 + 3: embed one probe input to **measure** the Nova 2 output dimension and log it. If `config.expected_dimension` is non-null and disagrees, raise `DimensionMismatchError`. If `s3vectors.get_index` reports the index is missing, create it with the measured dimension. This notebook is the only place `create_index` runs — neither the app nor the CDK provider ever creates the index.

In [ ]:
class DimensionMismatchError(Exception):
    """Raised when config.expected_dimension disagrees with the measured dim."""


# Probe: measure the model's native output dimension on a representative input.
probe_vector = embed(text="a fashion outfit", purpose=PURPOSE_INDEX)
measured_dim = len(probe_vector)
print(f"measured Nova 2 embedding dimension: {measured_dim}")

if EXPECTED_DIMENSION is not None and int(EXPECTED_DIMENSION) != measured_dim:
    raise DimensionMismatchError(
        f"config expected_dimension={EXPECTED_DIMENSION} but model emitted {measured_dim}"
    )

NOT_FOUND_CODES = {"NotFoundException", "ResourceNotFoundException", "NotFound"}

try:
    s3vectors.get_index(vectorBucketName=VECTOR_BUCKET, indexName=VECTOR_INDEX)
    print(f"index '{VECTOR_INDEX}' already exists; reusing it")
except ClientError as exc:
    code = exc.response.get("Error", {}).get("Code", "")
    if code not in NOT_FOUND_CODES:
        raise
    print(f"index '{VECTOR_INDEX}' not found; creating with dimension {measured_dim}")
    s3vectors.create_index(
        vectorBucketName=VECTOR_BUCKET,
        indexName=VECTOR_INDEX,
        dataType="float32",
        dimension=measured_dim,
        distanceMetric=DISTANCE_METRIC,
    )
    print("index created")

## 6. Ingest the catalog

For each dataset image: upload the original to `s3://{IMAGE_BUCKET}/catalog/<name>`, embed it with purpose `GENERIC_INDEX`, and `put_vectors` with the metadata carrying `s3_uri` + `name` (FR-2.3 — **no** base64 blob in the vector metadata).

In [ ]:
dataset_path = Path("Fashion-Dataset-Images-Western-Dress/WesternDress_Images")
image_paths = [
    p
    for p in dataset_path.iterdir()
    if p.is_file() and not p.name.startswith(".")
]

failed = []
for image_path in tqdm(image_paths, total=len(image_paths)):
    try:
        name = image_path.name
        catalog_key = f"catalog/{name}"
        with open(image_path, "rb") as fh:
            s3.put_object(Bucket=IMAGE_BUCKET, Key=catalog_key, Body=fh.read())
        catalog_uri = f"s3://{IMAGE_BUCKET}/{catalog_key}"

        vector = embed(image_b64=encode_image(image_path), purpose=PURPOSE_INDEX)
        vector_id = image_path.stem

        s3vectors.put_vectors(
            vectorBucketName=VECTOR_BUCKET,
            indexName=VECTOR_INDEX,
            vectors=[
                {
                    "key": vector_id,
                    "data": {"float32": vector},
                    "metadata": {"s3_uri": catalog_uri, "name": name},
                }
            ],
        )
    except Exception as exc:  # noqa: BLE001 - record and continue the batch
        print(f"failed on {image_path}: {exc}")
        failed.append(image_path)

print(f"Ingestion complete. Failed: {failed}")

## 7. Cleanup

The vector bucket, its index, and the image bucket are all torn down by `cdk destroy` (the custom resource best-effort deletes the index then the vector bucket). No manual cleanup is required here.